In [1]:
import spacy
from spacy import displacy

In [2]:
nlp = spacy.load("en_core_web_sm")

## Phần 2: Phân tích câu và trực quan hóa

In [3]:
text = 'Quick brown fox jumps over the lazy dog.'

In [4]:
doc = nlp(text)

### Trực quan hóa cây phụ thuộc

In [5]:
displacy.serve(doc, style="dep")

c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\spacy\displacy\__init__.py:106: UserWarning: [W011] It looks like you're calling displacy.serve from within a Jupyter notebook or a similar environment. This likely means you're already running a local web server, so there's no need to make displaCy start another one. Instead, you should be able to replace displacy.serve with displacy.render to show the visualization.
  warnings.warn(Warnings.W011)



Using the 'dep' visualizer
Serving on http://0.0.0.0:5000 ...

Shutting down server on port 5000.


- Từ gốc của câu là từ "jumps".
- Các từ phụ thuộc và mối quan hệ với từ "jumps" gồm:
    - "fox": nsubj (chủ ngữ của động từ).
    - "over": prep (giới từ).
- Từ "fox" là head của các từ "Quick" và "brown".

## Phần 3: Truy cập các thành phần trong cây phụ thuộc

In [6]:
text = 'Apple is looking at buying U.K. startup for $1 bililion'
doc = nlp(text)

In [10]:
print(f"{'TEXT':<12} | {'DEP':<10} | {'HEAD TEXT':<12} | {'HEAD POS':<8} | {'CHILDREN'}")
print("-" * 70)
for token in doc:
    children = [child.text for child in token.children]
    print(f"{token.text:<12} | {token.dep_:<10} | {token.head.text:<12} | {token.head.pos_:<8} | {children}")

TEXT         | DEP        | HEAD TEXT    | HEAD POS | CHILDREN
----------------------------------------------------------------------
Apple        | nsubj      | looking      | VERB     | []
is           | aux        | looking      | VERB     | []
looking      | ROOT       | looking      | VERB     | ['Apple', 'is', 'at']
at           | prep       | looking      | VERB     | ['buying']
buying       | pcomp      | at           | ADP      | ['startup']
U.K.         | nsubj      | startup      | VERB     | []
startup      | ccomp      | buying       | VERB     | ['U.K.', 'for']
for          | prep       | startup      | VERB     | ['bililion']
$            | nmod       | 1            | NUM      | []
1            | nummod     | bililion     | NOUN     | ['$']
bililion     | pobj       | for          | ADP      | ['1']


## Phần 4: Duyệt cây phụ thuộc để trích xuất thông tin

### Tìm chủ ngữ và tân ngữ của một động từ

In [11]:
text = "The cat chased the mouse and the dog watched them."
doc = nlp(text)

In [9]:
for token in doc:
    if token.pos_ == "VERB":
        verb = token.text
        subject = ""
        obj = ""
        
        for child in token.children:
            if child.dep_ == "nsubj":
                subject = child.text
            if child.dep_ == "dobj":
                obj = child.text

        if subject and obj:
            print(f"Found Triplet: ({subject}, {verb}, {obj})")

Found Triplet: (cat, chased, mouse)
Found Triplet: (dog, watched, them)


### Tìm các tính từ bổ nghĩa cho một danh từ

In [10]:
text = "The big, fluffy white cat is sleeping on the warm mat."
doc = nlp(text)

In [11]:
for token in doc:
    if token.pos_ == "NOUN":
        adjectives = []
        
        for child in token.children:
            if child.dep_ == "amod":
                adjectives.append(child.text)
        if adjectives:
            print(f"Danh từ '{token.text}' được bổ nghĩa bởi các tính từ: {adjectives}")

Danh từ 'cat' được bổ nghĩa bởi các tính từ: ['big', 'fluffy', 'white']
Danh từ 'mat' được bổ nghĩa bởi các tính từ: ['warm']


## Phần 5: Bài tập tự luyện

In [15]:
text = 'Quick brown fox jumps over the lazy dog.'
doc = nlp(text)

### Bài 1: Tìm động từ chính của câu

In [13]:
def find_main_verb(doc):
    for token in doc:
        if token.dep_ == "ROOT":
            return token
    
    return None

In [17]:
print('Main verb of text is:', find_main_verb(doc).text)

Main verb of text is: jumps


### Bài 2: Trích xuất các cụm danh từ (Noun Chunks)

In [24]:
def extract_noun_chunks(doc):
    noun_chunks = []

    for token in doc:
        if token.pos_ in ("NOUN", "PROPN", "PRON"):
            children = []

            for child in token.children:
                if child.dep_ in ("det", "amod", "compound", "poss", "nummod"):
                    children.append(child)

            chunk_tokens = children + [token]
            chunk_tokens = sorted(chunk_tokens, key=lambda x: x.i)

            chunk_text = " ".join([t.text for t in chunk_tokens])
            noun_chunks.append(chunk_text)

    return noun_chunks

In [25]:
print('Noun chunks in text is:', extract_noun_chunks(doc))

Noun chunks in text is: ['Quick brown fox', 'the lazy dog']


### Bài 3: Tìm đường đi ngắn nhất trong cây

In [28]:
def get_path_to_root(token):
    path = []
    current = token

    while True:
        path.append(current)
        if current.head == current:
            break
        current = current.head

    return path

In [30]:
token = doc[6]
print('Path from', token.text, 'to ROOT:', get_path_to_root(token))

Path from lazy to ROOT: [lazy, dog, over, jumps]
